In [91]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc

In [92]:
def create_ampl_instance(solver: str = "highs"):
    ampl = ampl_notebook(modules=[solver], license_uuid="default")
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl

1. La Química fabrica tres productos químicos: A,B y C. Estos productos se
obtienen por medio de 2 procesos de producción. El desarrollo del primer proceso durante 1 hora cuesta 40 USD y genera 3 unidades de A, una de B y y una de C. Efectuar el segundo proceso durante 1 hora cuesta 10 USD, y se obtienen una unidad de A y una de B. Para cumplir con las demandas de los clientes se tienen que producir todos los días por lo menos 40 unidades de A, 15 de B y 5 de C. Determinar un plan de producción diario, que minimice el costo de cumplir las demandas diarias de la Química

In [93]:
def resolver_control1_pregunta_1(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        var proceso_1 >= 0 integer;
        var proceso_2 >= 0 integer;

        minimize Cost:
          proceso_1 * 40 + proceso_2 * 10;

        subject to A_total:
          proceso_1 * 3 + proceso_2 * 1 >= 40;

        subject to B_total:
          proceso_1 * 1 + proceso_2 * 1 >= 15;

        subject to C_total:
          proceso_1 * 1 + proceso_2 * 0 >= 5;

        ''')
    ampl.solve(solver=solver)

    solution = {
        "proceso_1": int(round(ampl.get_value("proceso_1"))),
        "proceso_2": int(round(ampl.get_value("proceso_2"))),
        "cost": int(round(ampl.get_value("Cost"))),
        }
    return solution

In [94]:
ampl_result = resolver_control1_pregunta_1()
for clave, valor in ampl_result.items():
    print(f"{clave}: {valor}")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: proceso_1: 5
proceso_2: 25
cost: 450


2. Todas las semanas, Charcha puede comprar cantidades ilimitadas de materia prima a 6 dólares la libra. Cada libra de materia prima comprada se puede usar para elaborar el insumo 1 o el 2. Cada libra de materia prima rinde 2 oz del insumo 1, requiere 2 h. de tiempo de proceso e incurre en costos de proceso por 2 dólares. Cada libra de materia prima rinde 3 oz del insumo 2, requiere 2 h. de tiempo de proceso y sus costos de proceso son de 4 dólares. Se dispone de dos procesos de producción. El proceso 1 requiere 2 h., 2 oz de insumo 1 y 1 oz del insumo 2. Cuesta 1 dólar ejecutar el proceso 1. Cada vez que el proceso 1 se efectúa, se produce 1 oz del producto A y 1 oz de desecho líquido. Cada vez que el proceso 2 se efectúa, se requieren 3 h. de proceso, 2 oz del insumo 2 y 1 oz del insumo 1. El proceso 2 rinde 1 oz del producto B y 0.8 oz de desecho líquido. Los costos del proceso 2 son 8 dólares. Charcha puede tirar sus desechos líquidos en el río Port Charles, o bien, usar el desecho para elaborar el producto C o el producto D. De acuerdo con los reglamentos gubernamentales, Charcha tiene permitido derramar al río cuando mucho 1 000 oz a la semana. El costo por producir una onza del producto C es de 4 dólares, y se vende en 11 dólares. Se requiere una hora de tiempo de proceso, 2 oz del insumo 1 y 0.8 oz de desecho líquido para producir una onza del producto C. Cuesta 5 dólares fabricar una unidad del producto D y se vende en 7 dólares. Una hora de tiempo de proceso, 2 oz del insumo 2 y 1.2 oz de desecho líquido es lo que se requiere para fabricar una onza del producto D. Todas las semanas se venden, cuando mucho, 5 000 oz del producto A y 5 000 oz del producto B, pero la demanda semanal de los productos C y D es ilimitada. El producto A se vende a 18 dólares la onza y cada onza del producto B se vende en 24 dólares. Se dispone cada semana de 6 000 h. de tiempo de proceso. Formule un PL cuya solución le señale a Charcha cómo maximizar las utilidades semanales

In [95]:
def resolver_control1_pregunta_2(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        var materia_1 >= 0 integer;
        var materia_2 >= 0 integer;
        var producto_a >= 0 integer;
        var producto_b >= 0 integer;
        var producto_c >= 0 integer;
        var producto_d >= 0 integer;
        var Desecho_Rio >= 0 integer;

        maximize Profit:
          (18 * producto_a + 24 * producto_b + 11 * producto_c + 7 * producto_d)
          - (6 * (materia_1 + materia_2))
          - (2 * materia_1 + 4 * materia_2)
          - (1 * producto_a + 8 * producto_b)
          - (4 * producto_c + 5 * producto_d);

        subject to Balance_Insumo1:
            2 * producto_a + 1 * producto_b + 2 * producto_c <= 2 * materia_1;

        subject to Balance_Insumo2:
            1 * producto_a + 2 * producto_b + 2 * producto_d <= 3 * materia_2;

        subject to Balance_Desecho:
            1 * producto_a + 0.8 * producto_b == 0.8 * producto_c + 1.2 * producto_d + Desecho_Rio;

        subject to Tiempo_Total:
            2 * materia_1 + 2 * materia_2 + 2 * producto_a + 3 * producto_b + 1 * producto_c + 1 * producto_d <= 6000;

        subject to Limite_Rio:
            Desecho_Rio <= 1000;

        subject to Demanda_Max_A:
            producto_a <= 5000;

        subject to Demanda_Max_B:
            producto_b <= 5000;
        ''')
    ampl.solve(solver=solver)

    solution = {
        "materia_1": int(round(ampl.get_value("materia_1"))),
        "materia_2": int(round(ampl.get_value("materia_2"))),
        "Profit": int(round(ampl.get_value("Profit"))),
        "producto_a": int(round(ampl.get_value("producto_a"))),
        "producto_b": int(round(ampl.get_value("producto_b"))),
        "producto_c": int(round(ampl.get_value("producto_c"))),
        "producto_d": int(round(ampl.get_value("producto_d"))),
        "Desecho_Rio": int(round(ampl.get_value("Desecho_Rio")))
        }
    return solution

In [96]:
ampl_result = resolver_control1_pregunta_2()
for clave, valor in ampl_result.items():
    print(f"{clave}: {valor}")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: materia_1: 1351
materia_2: 388
Profit: 6364
producto_a: 1152
producto_b: 6
producto_c: 196
producto_d: 0
Desecho_Rio: 1000


3.	Usted ha sido designado administrador de la refinería de Melrose. Esta refinería produce gasolina y aceite combustible a partir del petróleo crudo. La gasolina se vende a 8 dólares el barril, y debe tener un 'grado' promedio de por lo menos 9. El aceite combustible se vende a 6 dólares el barril y debe tener un 'grado' de por lo menos 7. Se pueden vender, cuando mucho, 2 000 barriles de gasolina y 600 barriles de aceite combustible. El crudo que está por llegar puede ser procesado por medio de uno de tres métodos distintos. El rendimiento por barril y el costo por barril de cada método de proceso se proporcionan en la tabla 32. Por ejemplo, si se refina un barril del crudo que llega por el método 1, cuesta 3.40 dólares y da un rendimiento de 0.2 de barril de grado 6, 0.2 de barril de grado 8 y 0.6 de barril de grado 10. Antes de ser procesado para obtener gasolina y aceite combustible, los grados 6 y 8 podrían enviarse al desintegrador catalítico para mejorar su calidad. Por 1.30 dólares el barril, un barril de grado 6 puede ser fraccionado en 1 barril de grado 8. Por 2 dólares el barril, un barril de grado 8 se fracciona en un barril de grado 10. Cualquier crudo excedente, procesado o fraccionado, que ya no se pueda utilizar para aceite combustible o gasolina, se debe desechar a un costo de 0.20 de dólar por barril. Determine cómo maximizar la utilidad de la refinería

In [97]:
def resolver_control1_pregunta_3(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        var metodo_1 >= 0 integer;
        var metodo_2 >= 0 integer;
        var metodo_3 >= 0 integer;
        var mejora_6_a_8 >= 0 integer;
        var mejora_8_a_10 >= 0 integer;
        var gas_6 >= 0 integer;
        var gas_8 >= 0 integer;
        var gas_10 >= 0 integer;
        var aceite_6 >= 0 integer;
        var aceite_8 >= 0 integer;
        var aceite_10 >= 0 integer;
        var desecho_6 >= 0 integer;
        var desecho_8 >= 0 integer;
        var desecho_10 >= 0 integer;

        maximize Utilidad:
            8 * (gas_6 + gas_8 + gas_10) +
            6 * (aceite_6 + aceite_8 + aceite_10)
            - 3.4 * metodo_1 - 3.0 * metodo_2 - 2.6 * metodo_3
            - 1.3 * mejora_6_a_8 - 2.0 * mejora_8_a_10
            - 0.2 * (desecho_6 + desecho_8 + desecho_10);

        subject to Venta_Max_Gasolina:
            gas_6 + gas_8 + gas_10 <= 2000;

        subject to Venta_Max_Aceite:
            aceite_6 + aceite_8 + aceite_10 <= 600;

        subject to Calidad_Gasolina:
            -3 * gas_6 - 1 * gas_8 + 1 * gas_10 >= 0;

        subject to Calidad_Aceite:
            -1 * aceite_6 + 1 * aceite_8 + 3 * aceite_10 >= 0;

        subject to Balance_Grado_6:
            0.2 * metodo_1 + 0.3 * metodo_2 + 0.4 * metodo_3 == mejora_6_a_8 + gas_6 + aceite_6 + desecho_6;

        subject to Balance_Grado_8:
            0.2 * metodo_1 + 0.3 * metodo_2 + 0.4 * metodo_3 + mejora_6_a_8 == mejora_8_a_10 + gas_8 + aceite_8 + desecho_8;

        subject to Balance_Grado_10:
            0.6 * metodo_1 + 0.4 * metodo_2 + 0.2 * metodo_3 + mejora_8_a_10 == gas_10 + aceite_10 + desecho_10;
        ''')
    ampl.solve(solver=solver)

    solution = {
      "Utilidad_Neta_USD": int(round(ampl.get_value("Utilidad"))),

      "Compras en crudo": "",
      "Ejecutar_Metodo_1": int(round(ampl.get_value("metodo_1"))),
      "Ejecutar_Metodo_2": int(round(ampl.get_value("metodo_2"))),
      "Ejecutar_Metodo_3": int(round(ampl.get_value("metodo_3"))),

      "Mejoras": "",
      "Mejorar_Grado_6_a_8": int(round(ampl.get_value("mejora_6_a_8"))),
      "Mejorar_Grado_8_a_10": int(round(ampl.get_value("mejora_8_a_10"))),

      "Produccion de gasolina": "",
      "Gasolina_con_Grado_6": int(round(ampl.get_value("gas_6"))),
      "Gasolina_con_Grado_8": int(round(ampl.get_value("gas_8"))),
      "Gasolina_con_Grado_10": int(round(ampl.get_value("gas_10"))),

      "Produccion de aceite": "",
      "Aceite_con_Grado_6": int(round(ampl.get_value("aceite_6"))),
      "Aceite_con_Grado_8": int(round(ampl.get_value("aceite_8"))),
      "Aceite_con_Grado_10": int(round(ampl.get_value("aceite_10"))),

      "desechos": "",
      "Desecho_Total_Tirado": int(round((ampl.get_value("desecho_6"))
      + (ampl.get_value("desecho_8"))
      + (ampl.get_value("desecho_10"))))}
    return solution

In [98]:
ampl_result = resolver_control1_pregunta_3()
for clave, valor in ampl_result.items():
    print(f"{clave}: {valor}")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: Utilidad_Neta_USD: 11230
Compras en crudo: 
Ejecutar_Metodo_1: 0
Ejecutar_Metodo_2: 2400
Ejecutar_Metodo_3: 200
Mejoras: 
Mejorar_Grado_6_a_8: 500
Mejorar_Grado_8_a_10: 0
Produccion de gasolina: 
Gasolina_con_Grado_6: 0
Gasolina_con_Grado_8: 1000
Gasolina_con_Grado_10: 1000
Produccion de aceite: 
Aceite_con_Grado_6: 300
Aceite_con_Grado_8: 300
Aceite_con_Grado_10: 0
desechos: 
Desecho_Total_Tirado: 0


5. Un grupo de investigación de mercado necesita detectar por lo menos a 150 esposas, 120 esposos, 100 varones adultos solteros y 110 mujeres adultas solteras mediante una encuesta telefónica. Cuesta 2 dólares hacer una llamada en el día y (debido a los costos de mano de obra más altos) 5 dólares una llamada por la noche. Los resultados se dan en la tabla. Debido a que el personal es limitado, cuando mucho la mitad de todas las llamadas pueden ser nocturnas. Plantee un PL para minimizar el costo de completar la encuesta

In [99]:
def resolver_control1_pregunta_5(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        var llamadas_dia >= 0 integer;
        var llamadas_noche >= 0 integer;

        minimize Costo_Encuesta:
            2 * llamadas_dia + 5 * llamadas_noche;

        subject to Meta_Esposas:
            0.30 * llamadas_dia + 0.30 * llamadas_noche >= 150;

        subject to Meta_Esposos:
            0.10 * llamadas_dia + 0.30 * llamadas_noche >= 120;

        subject to Meta_Varones_Solteros:
            0.10 * llamadas_dia + 0.15 * llamadas_noche >= 100;

        subject to Meta_Mujeres_Solteras:
            0.10 * llamadas_dia + 0.20 * llamadas_noche >= 110;

        subject to Limite_Personal:
            llamadas_noche <= llamadas_dia;
        ''')
    ampl.solve(solver=solver)

    solution = {
        "Costo_Total_USD": int(round(ampl.get_value("Costo_Encuesta"))),
        "Llamadas": "",
        "Llamadas_a_realizar_de_DIA": int(round(ampl.get_value("llamadas_dia"))),
        "Llamadas_a_realizar_de_NOCHE": int(round(ampl.get_value("llamadas_noche"))),
        "Se esperan conectados": "",
        "Esposas_Contactadas": int(round(0.3 * ampl.get_value("llamadas_dia") + 0.3 * ampl.get_value("llamadas_noche"))),
        "Esposos_Contactados": int(round(0.1 * ampl.get_value("llamadas_dia") + 0.3 * ampl.get_value("llamadas_noche"))),
        "Varones_Solteros": int(round(0.1 * ampl.get_value("llamadas_dia") + 0.15 * ampl.get_value("llamadas_noche"))),
        "Mujeres_Solteras": int(round(0.1 * ampl.get_value("llamadas_dia") + 0.2 * ampl.get_value("llamadas_noche")))
        }
    return solution

In [100]:
ampl_result = resolver_control1_pregunta_5()
for clave, valor in ampl_result.items():
    print(f"{clave}: {valor}")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: Costo_Total_USD: 2300
Llamadas: 
Llamadas_a_realizar_de_DIA: 900
Llamadas_a_realizar_de_NOCHE: 100
Se esperan conectados: 
Esposas_Contactadas: 300
Esposos_Contactados: 120
Varones_Solteros: 105
Mujeres_Solteras: 110


6.	Brady Corporation fabrica alacenas. Requiere cada semana 90 000 pies cúbicos de tablones. La compañía puede conseguir madera de dos maneras: primero, la podría comprar con un proveedor y secarla en el horno del proveedor. Segundo, podría cortar troncos en sus propios terrenos, cortarlos en tablones en su aserradero y, por último, secarlos en su propio horno. Compra y Calidad de Tablones: Brady puede comprar tablones grado 1 o grado 2. Los tablones grado 1 cuestan 3 dólares por pie cúbico, y cuando se secan rinden 0.7 pies cúbicos de madera útil. Los tablones grado 2 cuestan 7 dólares el pie cúbico, y luego de secarlos rinden 0.9 pies cúbicos de madera útil.
Procesamiento de Troncos Propios: A la compañía le cuesta 3 dólares cortar los troncos. Después de cortar y secar un tronco, éste rinde 0.8 pies cúbicos de tablones. Costos de Operación: Brady gasta 4 dólares por pie cúbico de tablones secados. Además, cuesta 2.50 dólares por pie cúbico de troncos enviados al aserradero. Capacidades y Restricciones: El aserradero puede procesar cada semana hasta 35 000 pies cúbicos de tablones. Se pueden comprar cada semana hasta 40 000 pies cúbicos de tablones grado 1 y hasta 60 000 pies cúbicos del grado 2. Se dispone cada semana de 40 h para secar los tablones. Tiempos de Secado: El tiempo que se requiere para secar 1 pie cúbico de madera es el siguiente: Grado 1: 1.2 segundos (s).Grado 2: 0.8 s. Troncos: 1.3 s. Determine un PL (Programación Lineal) que ayude a Brady a minimizar el costo a la semana por cumplir con la demanda de tablones procesados


In [101]:
def resolver_control1_pregunta_6(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        var grado_1 >= 0;
        var grado_2 >= 0;
        var troncos >= 0;

        minimize Costo_Total:
            (3 + 4) * grado_1 + (7 + 4) * grado_2 + (3 + 2.5 + 4) * troncos;

        subject to Demanda_madera_util:
            0.7 * grado_1 + 0.9 * grado_2 + 0.8 * troncos >= 90000;

        subject to Limite_compra_G1:
            grado_1 <= 40000;

        subject to Limite_compra_G2:
            grado_2 <= 60000;

        subject to Limite_aserradero_propio:
            troncos <= 35000;

        subject to Tiempo_secado_segundos:
            1.2 * grado_1 + 0.8 * grado_2 + 1.3 * troncos <= 144000;
        ''')
    ampl.solve(solver=solver)

    solution = {
        "Costo_Minimo_USD": round(ampl.get_value("Costo_Total") or 0, 2),

        "Cantidad de pies cubicos": "",
        "Comprar_Grado_1": round(ampl.get_value("grado_1")),
        "Comprar_Grado_2": round(ampl.get_value("grado_2")),
        "Procesar_Troncos_Propios": round(ampl.get_value("troncos")),

        "Tiempo y Cantidad": "",
        "Madera_Util_Generada": round(0.7 * ampl.get_value("grado_1") + 0.9 * ampl.get_value("grado_2") + 0.8 * ampl.get_value("troncos"), 2),
        "Segundos_de_Horno_Usados": round(1.2 * ampl.get_value("grado_1") + 0.8 * ampl.get_value("grado_2") + 1.3 * ampl.get_value("troncos"), 2)
        }
    return solution

In [102]:
ampl_result = resolver_control1_pregunta_6()
for clave, valor in ampl_result.items():
    print(f"{clave}: {valor}")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: Costo_Minimo_USD: 1028055.56
Cantidad de pies cubicos: 
Comprar_Grado_1: 40000
Comprar_Grado_2: 37778
Procesar_Troncos_Propios: 35000
Tiempo y Cantidad: 
Madera_Util_Generada: 90000.0
Segundos_de_Horno_Usados: 123722.22


7. Un centro de reciclaje industrial utiliza dos chatarras de aluminio, A y B, para producir una aleación especial. La chatarra A contiene 6% de aluminio, 3% de silicio, y 4% de carbón. La chatarra B contiene 3% de aluminio, 6% de silicio, y 3% de carbón. Los costos por tonelada de las chatarras A y B son de $100 y $80, respectivamente. Las especificaciones de la aleación especial requieren que (1) el contenido de aluminio debe ser mínimo de 3% y máximo de 6%; (2) el contenido de silicio debe ser de entre 3 y 5%, y (3) el contenido de carbón debe ser de entre 3 y 7%. Determine la mezcla óptima de las chatarras que deben usarse para producir 1000 toneladas de la aleación.

In [103]:
def resolver_control1_pregunta_7(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        var chatarra_A >= 0;
        var chatarra_B >= 0;

        minimize Costo_Total:
            100 * chatarra_A + 80 * chatarra_B;

        subject to Peso_Exacto:
            chatarra_A + chatarra_B == 1000;

        subject to Aluminio_Minimo:
            0.06 * chatarra_A + 0.03 * chatarra_B >= 0.03 * 1000;

        subject to Aluminio_Maximo:
            0.06 * chatarra_A + 0.03 * chatarra_B <= 0.06 * 1000;

        subject to Silicio_Minimo:
            0.03 * chatarra_A + 0.06 * chatarra_B >= 0.03 * 1000;

        subject to Silicio_Maximo:
            0.03 * chatarra_A + 0.06 * chatarra_B <= 0.05 * 1000;

        subject to Carbon_Minimo:
            0.04 * chatarra_A + 0.03 * chatarra_B >= 0.03 * 1000;

        subject to Carbon_Maximo:
            0.04 * chatarra_A + 0.03 * chatarra_B <= 0.07 * 1000;
        ''')
    ampl.solve(solver=solver)

    solution = {
      "Costo_Total_USD": round(ampl.get_value("Costo_Total")),
      "Toneladas_Chatarra_A": round(ampl.get_value("chatarra_A")),
      "Toneladas_Chatarra_B": round(ampl.get_value("chatarra_B"))
      }
    return solution

In [104]:
ampl_result = resolver_control1_pregunta_7()
for clave, valor in ampl_result.items():
    print(f"{clave}: {valor}")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: Costo_Total_USD: 86667
Toneladas_Chatarra_A: 333
Toneladas_Chatarra_B: 667
